In [11]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

In [12]:
mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto') #fetching MNIST dataset

# Prepare features and labels
X = mnist.data.values.astype('float32') / 255.0
y = mnist.target.values.astype('int')

# Split 60k train / 10k test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=60000, test_size=10000, random_state=42)
print(f"Data ready: {X_train.shape[0]} train samples, {X_test.shape[0]} test samples.\n")


Data ready: 60000 train samples, 10000 test samples.



In [13]:
T_checkpoints = [0.1, 1, 2, 3, 4, 10, 30]
methods_rows = [
    'Vote', 'Avg. (unnorm)', 'Avg. (norm)', 
    'Last (unnorm)', 'Last (norm)', 
    'Rand. (unnorm)', 'Rand. (norm)', 
    'SupVec', 'Mistake'
]

table1 = pd.DataFrame(index=pd.MultiIndex.from_product([[1, 2, 3], methods_rows], names=['d', 'Method']), columns=T_checkpoints)
table2 = pd.DataFrame(index=pd.MultiIndex.from_product([[4, 5, 6], methods_rows], names=['d', 'Method']), columns=T_checkpoints)

In [ ]:
for d in [1, 2, 3, 4, 5, 6]:
    print(f"\n=========================================")
    print(f" Starting Polynomial Degree d = {d} ")
    print(f"=========================================")
    
    for T in T_checkpoints:
        start_time = time.time()
        
        epochs = 1 if T == 0.1 else int(T)
        limit = 6000 if T == 0.1 else 60000
        
        X_tr = X_train[:limit]
        y_tr = y_train[:limit]
        total_steps = epochs * len(X_tr)
        
        total_mistakes = 0
        supvecs = set()
        
        # Initialize prediction score matrices
        scores_vote = np.zeros((10000, 10))
        scores_avg_u = np.zeros((10000, 10))
        scores_avg_n = np.zeros((10000, 10))
        scores_last_u = np.zeros((10000, 10))
        scores_last_n = np.zeros((10000, 10))
        
        # Memory storage for Rand sampling later
        memory_dots_u = []
        memory_dots_n = []
        memory_C = []
        
        for c in range(10):
            y_bin = np.where(y_tr == c, 1, -1)
            
            # ---------------------------------------------------------
            # LINEAR FORMULATION (d=1)
            # ---------------------------------------------------------
            if d == 1:
                V = [np.zeros(X_tr.shape[1])]
                C = [0]
                
                for epoch in range(epochs):
                    for i in range(len(X_tr)):
                        x_i, y_i = X_tr[i], y_bin[i]
                        y_hat = 1 if np.dot(V[-1], x_i) >= 0 else -1
                        
                        if y_hat == y_i:
                            C[-1] += 1
                        else:
                            V.append(V[-1] + y_i * x_i)
                            C.append(1)
                            total_mistakes += 1
                            supvecs.add(i)
                            
                # Vectorized calculations
                V_mat = np.array(V)
                C_vec = np.array(C)
                norms = np.linalg.norm(V_mat, axis=1)
                norms[norms == 0] = 1.0 
                V_mat_norm = V_mat / norms[:, np.newaxis]
                
                dots_u = np.dot(X_test, V_mat.T)
                dots_n = np.dot(X_test, V_mat_norm.T)

            # ---------------------------------------------------------
            # KERNEL FORMULATION (d>1)
            # ---------------------------------------------------------
            else:
                mistakes = []
                C = [0]
                v_norms = [0.0]
                v_norm_sq = 0.0
                
                for epoch in range(epochs):
                    for i in range(len(X_tr)):
                        x_i, y_i = X_tr[i], y_bin[i]
                        
                        if len(mistakes) == 0:
                            v_dot_x = 0
                        else:
                            M_x = np.array([m[0] for m in mistakes])
                            M_y = np.array([m[1] for m in mistakes])
                            v_dot_x = np.sum(M_y * ((np.dot(M_x, x_i) + 1.0) ** d))
                                
                        y_hat = 1 if v_dot_x >= 0 else -1
                        
                        if y_hat == y_i:
                            C[-1] += 1
                        else:
                            # Exact Kernel Norm tracking: ||v_{j+1}||^2 = ||v_j||^2 + 2y(v_j * x) + ||x||^2
                            x_i_norm_sq = (np.dot(x_i, x_i) + 1.0) ** d
                            v_norm_sq = v_norm_sq + 2 * y_i * v_dot_x + x_i_norm_sq
                            
                            mistakes.append((x_i, y_i))
                            C.append(1)
                            v_norms.append(np.sqrt(max(0, v_norm_sq)))
                            total_mistakes += 1
                            supvecs.add(i)
                            
                # Reconstruct testing state exactly
                C_vec = np.array(C)
                if len(mistakes) > 0:
                    M_x = np.array([m[0] for m in mistakes])
                    M_y = np.array([m[1] for m in mistakes])
                    
                    K_matrix = (np.dot(X_test, M_x.T) + 1.0) ** d
                    K_weighted = K_matrix * M_y
                    
                    # dots_u equivalent to v_j * x for all test cases
                    dots_u = np.hstack([np.zeros((10000, 1)), np.cumsum(K_weighted, axis=1)])
                    
                    norms = np.array(v_norms)
                    norms[norms == 0] = 1.0
                    dots_n = dots_u / norms
                else:
                    dots_u = np.zeros((10000, 1))
                    dots_n = np.zeros((10000, 1))

            # --- POPULATE DETERMINISTIC PREDICTION MATRICES ---
            signs = np.where(dots_u >= 0, 1, -1)
            
            scores_vote[:, c] = np.dot(signs, C_vec)
            scores_avg_u[:, c] = np.dot(dots_u, C_vec)
            scores_avg_n[:, c] = np.dot(dots_n, C_vec)
            scores_last_u[:, c] = dots_u[:, -1]
            scores_last_n[:, c] = dots_n[:, -1]
            
            # Store histories to memory for Rand calculation
            memory_dots_u.append(dots_u)
            memory_dots_n.append(dots_n)
            memory_C.append(C_vec)

        # ---------------------------------------------------------
        # EXACT RANDOM TIME SLICE EVALUATION (RAND. METHOD)
        # ---------------------------------------------------------
        num_samples = 100 # Simulating 100 random global time slices
        random_steps = np.random.randint(0, total_steps, size=num_samples)
        
        j_indices = np.zeros((10, num_samples), dtype=int)
        for c in range(10):
            C_cumsum = np.cumsum(memory_C[c])
            # Finds the exact perceptron active at global time 'r'
            j_indices[c] = np.searchsorted(C_cumsum, random_steps, side='right')
            j_indices[c] = np.clip(j_indices[c], 0, len(memory_C[c]) - 1)
            
        errors_rand_u = 0
        errors_rand_n = 0
        
        for s_idx in range(num_samples):
            temp_scores_u = np.zeros((10000, 10))
            temp_scores_n = np.zeros((10000, 10))
            
            for c in range(10):
                active_j = j_indices[c, s_idx]
                temp_scores_u[:, c] = memory_dots_u[c][:, active_j]
                temp_scores_n[:, c] = memory_dots_n[c][:, active_j]
                
            errors_rand_u += np.sum(np.argmax(temp_scores_u, axis=1) != y_test)
            errors_rand_n += np.sum(np.argmax(temp_scores_n, axis=1) != y_test)
            
        err_rand_u = (errors_rand_u / (num_samples * 10000)) * 100
        err_rand_n = (errors_rand_n / (num_samples * 10000)) * 100

        # Calculate standard errors
        err_vote = (1 - accuracy_score(y_test, np.argmax(scores_vote, axis=1))) * 100
        err_avg_u = (1 - accuracy_score(y_test, np.argmax(scores_avg_u, axis=1))) * 100
        err_avg_n = (1 - accuracy_score(y_test, np.argmax(scores_avg_n, axis=1))) * 100
        err_last_u = (1 - accuracy_score(y_test, np.argmax(scores_last_u, axis=1))) * 100
        err_last_n = (1 - accuracy_score(y_test, np.argmax(scores_last_n, axis=1))) * 100
        
        # Populate Target Table
        target_table = table1 if d <= 3 else table2
        
        target_table.loc[(d, 'Vote'), T] = round(err_vote, 1)
        target_table.loc[(d, 'Avg. (unnorm)'), T] = round(err_avg_u, 1)
        target_table.loc[(d, 'Avg. (norm)'), T] = round(err_avg_n, 1)
        target_table.loc[(d, 'Last (unnorm)'), T] = round(err_last_u, 1)
        target_table.loc[(d, 'Last (norm)'), T] = round(err_last_n, 1)
        target_table.loc[(d, 'Rand. (unnorm)'), T] = round(err_rand_u, 1)
        target_table.loc[(d, 'Rand. (norm)'), T] = round(err_rand_n, 1)
        target_table.loc[(d, 'SupVec'), T] = len(supvecs)
        target_table.loc[(d, 'Mistake'), T] = total_mistakes
        
        print(f"  T={T:<4} | Mistakes: {total_mistakes:<6} | Test Error: {err_vote:.2f}%  ({time.time() - start_time:.1f}s)")


 Starting Polynomial Degree d = 1 
  T=0.1  | Mistakes: 3232   | Test Error: 11.45%  (1.8s)
  T=1    | Mistakes: 25262  | Test Error: 9.04%  (8.8s)
  T=2    | Mistakes: 47995  | Test Error: 8.80%  (17.8s)
  T=3    | Mistakes: 70212  | Test Error: 8.68%  (26.1s)


In [ ]:
print("\n=================== TABLE 1 (d=1, 2, 3) ===================")
display(table1.fillna('-'))

print("\n=================== TABLE 2 (d=4, 5, 6) ===================")
display(table2.fillna('-'))